# VideoFake su Runpod

Ambiente e modelli restano in `/workspace/videofake-live`. Gli step scaricano l'ultimo codice dalla repo a ogni esecuzione.

**Se l'installazione è già stata completata via SSH: esegui 0, poi 4 e 5.** Lo step 3 reinstalla/verifica l'ambiente solo quando serve.

Carica foto e video nella cartella `/workspace/input` usando il file browser di JupyterLab. Imposta i percorsi nello step 4. Il risultato viene salvato in `/workspace/output/out.mp4`.

Durante lo step 5 vedi fase, avanzamento, frame elaborati e stima temporale. Un riquadro rosso indica errore; verde indica completamento.

Il Pod consuma credito finché è acceso, anche quando non elabori. Dopo aver scaricato i risultati, fermalo dalla console Runpod. Lo storage resta a pagamento da fermo.


In [ ]:
import os
os.environ.setdefault('VIDEOFAKE_HOME', '/workspace/videofake-live')
os.environ.setdefault('UV_PYTHON_INSTALL_DIR', '/workspace/videofake-live/python')
os.environ.setdefault('UV_CACHE_DIR', '/workspace/videofake-live/uv-cache')
os.environ.setdefault('PIP_NO_CACHE_DIR', '1')
#@title 0. Collega Colab al codice aggiornato su GitHub
import json
import time
import types
import urllib.request

def _vf_step(step, **config):
    global VF_DEBUG
    revision = 'non disponibile'
    try:
        api = 'https://api.github.com/repos/LucaLorenz/VideoFake/git/ref/heads/main'
        request = urllib.request.Request(api + '?t=' + str(time.time_ns()),
            headers={'User-Agent': 'VideoFake-Colab', 'Cache-Control': 'no-cache'})
        with urllib.request.urlopen(request, timeout=30) as response:
            revision = json.load(response)['object']['sha']
        url = f'https://raw.githubusercontent.com/LucaLorenz/VideoFake/{revision}/colab_runner.py'
        with urllib.request.urlopen(url, timeout=30) as response:
            code = response.read().decode('utf-8')
        runner = types.ModuleType('videofake_live_runner')
        exec(compile(code, 'colab_runner.py@' + revision[:12], 'exec'), runner.__dict__)
        VF_DEBUG = runner.run_step(step, revision, config)
    except Exception as error:
        VF_DEBUG = {'revision': revision, 'step': step, 'status': 'error',
                    'error': f'{type(error).__name__}: {error}'}
        print('DEBUG_BEGIN\n' + json.dumps(VF_DEBUG, ensure_ascii=False, indent=2) + '\nDEBUG_END')
    return VF_DEBUG

VF_DEBUG = _vf_step('diagnose')


In [ ]:
#@title 1. Scarica applicazione
VF_DEBUG = _vf_step("repo")


In [ ]:
#@title 2. Prepara cartelle
VF_DEBUG = _vf_step("folders")


In [ ]:
#@title 3. Installa dipendenze e verifica GPU
VF_DEBUG = _vf_step("setup")


In [ ]:
#@title 4. Imposta file di ingresso e uscita
source = "/workspace/input/face.jpg" #@param {type:"string"}
target = "/workspace/input/video.mp4" #@param {type:"string"}
output = "/workspace/output/out.mp4" #@param {type:"string"}
print('Percorsi impostati. Carica i tuoi file oppure usa gli esempi del progetto.')


In [ ]:
#@title 5. Esegui usando l'ultimo codice su GitHub
Device = "cuda" #@param ["cuda", "cpu"]
Processor = "face_swapper face_enhancer" #@param ["face_swapper face_enhancer", "face_swapper", "face_enhancer"]
VideoEncoder = "libx264" #@param ["libx264", "libx265", "libvpx-vp9"]
VideoQuality = "18" #@param {type:"string"}
KeepFPS = True #@param {type:"boolean"}
KeepAudio = True #@param {type:"boolean"}
KeepFrames = False #@param {type:"boolean"}
ManyFaces = True #@param {type:"boolean"}
VF_DEBUG = _vf_step('run', source=source, target=target, output=output,
    Device=Device, Processor=Processor, VideoEncoder=VideoEncoder, VideoQuality=VideoQuality,
    KeepFPS=KeepFPS, KeepAudio=KeepAudio, KeepFrames=KeepFrames, ManyFaces=ManyFaces)


In [ ]:
#@title 6. Mostra ultimo rapporto breve
print('DEBUG_BEGIN\n' + json.dumps(VF_DEBUG, ensure_ascii=False, indent=2) + '\nDEBUG_END')
